In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from firm3d.field.boozermagneticfield import (
    BoozerRadialInterpolant,
    InterpolatedBoozerField,
)
from firm3d.field.trajectory_helpers import TrappedPoincare
from firm3d.util.constants import (
    ALPHA_PARTICLE_CHARGE,
    ALPHA_PARTICLE_MASS,
    FUSION_ALPHA_PARTICLE_ENERGY,
)
from firm3d.util.functions import proc0_print, setup_logging
from firm3d.util.mpi import comm_size, comm_world, verbose

# Can comment out these inputs below if call_DESC==False
from desc.ResonanceOpt.TRObj_func import TrappedResonanceObj
import desc.io
import jax.numpy as jnp

In [ ]:
#######################
# COMMON USER INPUTS #
# firm3d setup
boozmn_filename = "../inputs/boozmn_equil_Helios_E0092_DESC_fixed.nc" # make sure this matches the DESC input
neta_poinc = 20  # Number of eta initial conditions for poincare
ns_poinc = 25  # Number of s initial conditions for poincare
Nmaps = 2500  # Number of Poincare return maps to compute
modBin = 6.55 # T
tmax = 1e-2

# DESC setup
call_DESC = True # if this is False, can comment out other lines in "DESC setup"
eq = desc.io.load("../inputs/equil_Helios_E0092_DESC_fixed.h5") # make sure this matches the firm3d input
rhos_desc = (np.linspace(0.1,0.9,100))**(1/2) # rho = sqrt(s)
alphas_desc = np.linspace(0,2*np.pi,3)
KE_frac_desc = np.array([1]) #did 0.001 before
N=0 # QA
#######################

In [ ]:
# Additional setup

Ekin = FUSION_ALPHA_PARTICLE_ENERGY * KE_frac_desc[0]
charge = ALPHA_PARTICLE_CHARGE
mass = ALPHA_PARTICLE_MASS

resolution = 48  # Resolution for field interpolation
ns_interp = resolution  # number of radial grid points for interpolation
ntheta_interp = resolution  # number of poloidal grid points for interpolation
nzeta_interp = resolution  # number of toroidal grid points for interpolation
order = 3  # order for interpolation
tol = 1e-8  # Tolerance for ODE solver
s_mirror = 0.5  # flux surface for mirroring
theta_mirror = np.pi / 2  # poloidal angle for mirroring
zeta_mirror = 0
helicity_M = 1  # helicity of field strength contours
helicity_N = N # =0 for QA
degree = 3  # Degree for Lagrange interpolation


# Setup logging to redirect output to file
#setup_logging(f"stdout_trapped_map_{resolution}_{comm_size}.txt")

In [1]:
# Poincare plotting
time1 = time.time()

bri = BoozerRadialInterpolant(boozmn_filename, order, no_K=True, comm=comm_world)

field = InterpolatedBoozerField(
    bri,
    degree,
    ns_interp=ns_interp,
    ntheta_interp=ntheta_interp,
    nzeta_interp=nzeta_interp,
)

poinc = TrappedPoincare(
    field,
    helicity_M,
    helicity_N,
    s_mirror,
    theta_mirror,
    zeta_mirror,
    mass,
    charge,
    Ekin,
    modBin=modBin,
    ns_poinc=ns_poinc,
    neta_poinc=neta_poinc,
    Nmaps=Nmaps,
    comm=comm_world,
    solver_options={"reltol": tol, "abstol": tol, "axis": 0},
    tmax=tmax,
)

time2 = time.time()
proc0_print("poincare time: ", time2 - time1)
ax = poinc.plot_poincare(ax=ax)

NameError: name 'time' is not defined

In [ ]:
# Run DESC objective function
pitch_invs_desc = jnp.array([modBin])
out = TrappedResonanceObj(eq,rhos_desc,pitch_invs_desc,KE_frac_desc,alphas_desc,N)
obj_val = out['obj'][:,0,0] # Only look at rho, for one pitch, for one energy
s_obj = np.linspace(0.1,0.9,len(obj_val))

In [ ]:
# Plotting the objective function over the Poincare plot
fig, ax = plt.subplots()
Y = s_obj
X = np.linspace(0,2*np.pi,5)
Z = np.transpose(np.tile(obj_val, (5, 1)))
cs = ax.contourf(X,Y,Z,cmap='Blues')
fig.colorbar(cs, ax=ax)

ax = poinc.plot_poincare(ax=ax)
ax.figure.savefig('poincare_objective_overlay.png')